In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from src.downloadData import download_dataset_csvs
summary = download_dataset_csvs(
    output_dir=repo_root /"data" /"raw",
    progress=True,
)

print(f"downloaded={len(summary.downloaded_paths)}, skipped={len(summary.skipped_paths)}")

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

import os
import sys
import shutil

python_exe = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exe
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exe

spark = (
    SparkSession.builder
    .appName("EEG_Schizoprenia")
    .master("local[*]")
    .config("spark.pyspark.python", python_exe)
    .config("spark.pyspark.driver.python", python_exe)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark started:", spark.version)
print("Python:", sc.pythonExec)

In [ ]:
base_path = ".small/raw"
data_paths = list(Path(base_path).rglob("*/*.csv"))
print(f"Found {len(data_paths)} files. " \
      f"From `{data_paths[0]}` to `{data_paths[-1]}`")

In [ ]:
column_names = spark.read.csv(r".\data\raw\columnLabels.csv",
                              header=True).columns
print("List of column labels:")
for i, column in enumerate(column_names, start=1):
    print(f"  {i}. {column}" )

In [ ]:
df = spark.read.csv(r".small\raw\*\*.csv", 
                    inferSchema=True, # infer data types
                    header=False)
df = df.toDF(*column_names) 

row_count = df.count()
print(f"Read {row_count:,} records")

df.show(5)

In [ ]:
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array

no_normalize = ["subject", "trial", "condition", "sample"]
normalize_cols = [c for c in df.columns if c not in no_normalize]

assembler = VectorAssembler(inputCols=normalize_cols, outputCol="features")
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=True
)
df_vec = assembler.transform(df)
df_vec = scaler.fit(df_vec).transform(df_vec)

normalized_df = df_vec.select(
    *no_normalize,
    *[vector_to_array("scaledFeatures")[i].alias(c)for i, c in enumerate(normalize_cols)]
)

normalized_df.show(5)

In [ ]:
normalized_df.createOrReplaceTempView("normalized_table")

# I asked GPT how to make my manual SAX code faster and 
# it said turn "PAA -> SAX -> Concat" into one SQL query  

# My old code that uses df.groupBy is in commit d5312bc1e5f31688c806054d731e824a8f844779

sax_exprs = [
    f"""
    CASE 
        WHEN AVG({c}) < -0.67 THEN 'a'
        WHEN AVG({c}) < 0 THEN 'b'
        WHEN AVG({c}) < 0.67 THEN 'c'
        ELSE 'd'
    END AS {c}_sax
    """
    for c in normalize_cols
]

final_exprs = [
    f"""
    CONCAT_WS(
        '',
        TRANSFORM(
            SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT('g', group_id, 'v', {c}_sax))),
            x -> x.v
        )
    ) AS {c}_sax
    """
    for c in normalize_cols
]

query = f"""
WITH base AS (
    SELECT
        subject,
        trial,
        condition,
        sample,
        NTILE(30) OVER (
            PARTITION BY subject, trial, condition
            ORDER BY sample
        ) - 1 AS group_id,
        {", ".join(normalize_cols)}
    FROM normalized_table
),

paa_sax AS (
    SELECT
        subject,
        trial,
        condition,
        group_id,
        {", ".join(sax_exprs)}
    FROM base
    GROUP BY subject, trial, condition, group_id
)

SELECT
    subject,
    trial,
    condition,
    {", ".join(final_exprs)}
FROM paa_sax
GROUP BY subject, trial, condition
"""

sax_df = spark.sql(query)
sax_df.show(5)

In [ ]:
from pyspark.sql.functions import avg, max, min, stddev

group_cols = ["subject", "trial", "condition"]
other_cols = [col_name for col_name in df.columns if col_name not in group_cols]
other_cols.remove("sample")

grp_functions = [(avg, "avg"),(max, "max"),(min, "min"),(stddev, "stddev")]

agg_df = df.groupBy(group_cols).agg(
                *[f(c).alias(f"{c}_{name}") for c in other_cols for f, name in grp_functions]
            ).orderBy(group_cols)
agg_df.show(15)

# In here I wanted to add mode as well but since mode is a heavy process, it caused memory errors
# I also tried adding q1, median, and q3 but it was also too heavy for our local machines

In [ ]:
# join the two dfs
joined_df = agg_df.join(
    sax_df,
    on=group_cols,
    how="inner"
)

ordered_cols = group_cols.copy()
for c in other_cols:
    ordered_cols.extend([
        f"{c}_avg",
        f"{c}_max",
        f"{c}_min",
        f"{c}_stddev",
        f"{c}_sax"
    ])

agg_sax_df = joined_df.select(*ordered_cols).orderBy(*group_cols)

agg_sax_df.show(5)

In [ ]:
agg_sax_df.coalesce(1).write.mode("overwrite").option("header", True).csv("data/processed/1-4_csv_agg")

In [ ]:
spark.stop()